# Meta-learning paper plots

This notebook generates the 2x2 figure for the four meta-learning tasks.
It will reuse cached `metrics.json` when present and compute them otherwise.

In [1]:
!which python

/mnt/nlp/scratch/home/mpanwar/miniconda3/bin/python


In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt

REPO_ROOT = Path.cwd().resolve().parent
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

os.environ.setdefault("HF_HOME", "/tmp/hf-home")

In [ ]:
from plot_meta_paper_figure import TASK_SPECS, get_task_metrics, plot_task, resolve_run_path

## Configure runs

Leave a value as `None` to automatically use the most recent run directory for that task.
Set `EVAL_NAME = "maml"` for meta-learning eval or `"standard"` for the usual ICL curves.

In [ ]:
EVAL_NAME = "maml"

LINEAR_RUN = None
SPARSE_RUN = None
TREE_RUN = None
RELU_RUN = None

In [ ]:
run_paths = {
    "linear_regression": resolve_run_path(REPO_ROOT, "linear_regression", LINEAR_RUN),
    "sparse_linear_regression": resolve_run_path(REPO_ROOT, "sparse_linear_regression", SPARSE_RUN),
    "decision_tree": resolve_run_path(REPO_ROOT, "decision_tree", TREE_RUN),
    "relu_2nn_regression": resolve_run_path(REPO_ROOT, "relu_2nn_regression", RELU_RUN),
}

run_paths

## Compute or load metrics

In [ ]:
task_metrics = {
    task_name: get_task_metrics(run_path, EVAL_NAME)
    for task_name, run_path in run_paths.items()
}

sorted(task_metrics.keys())

## Plot figure

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9), constrained_layout=True)
ordered_tasks = [
    "linear_regression",
    "sparse_linear_regression",
    "decision_tree",
    "relu_2nn_regression",
]

for ax, task_name in zip(axes.flat, ordered_tasks):
    plot_task(ax, task_name, task_metrics[task_name])

fig

## Save outputs

In [ ]:
output_dir = REPO_ROOT / "plots"
output_dir.mkdir(parents=True, exist_ok=True)

png_path = output_dir / f"meta_paper_figure_{EVAL_NAME}.png"
pdf_path = output_dir / f"meta_paper_figure_{EVAL_NAME}.pdf"

fig.savefig(png_path, dpi=300, bbox_inches="tight")
fig.savefig(pdf_path, bbox_inches="tight")

print("Saved:", png_path)
print("Saved:", pdf_path)